In [5]:
import pandas as pd


In [6]:
path = "/Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/traficom_datasets/TieliikenneAvoinData_31_12_2025.csv"

df = pd.read_csv(path, sep=";", encoding="latin1", low_memory=False)

In [7]:
print(df.shape)
print(df.columns.tolist())
print(df.info())

(5122260, 41)
['ajoneuvoluokka', 'ensirekisterointipvm', 'ajoneuvoryhma', 'ajoneuvonkaytto', 'variantti', 'versio', 'kayttoonottopvm', 'vari', 'ovienLukumaara', 'korityyppi', 'ohjaamotyyppi', 'istumapaikkojenLkm', 'omamassa', 'teknSuurSallKokmassa', 'tieliikSuurSallKokmassa', 'ajonKokPituus', 'ajonLeveys', 'ajonKorkeus', 'kayttovoima', 'iskutilavuus', 'suurinNettoteho', 'sylintereidenLkm', 'ahdin', 'sahkohybridi', 'sahkohybridinluokka', 'merkkiSelvakielinen', 'mallimerkinta', 'vaihteisto', 'vaihteidenLkm', 'kaupallinenNimi', 'voimanvalJaTehostamistapa', 'tyyppihyvaksyntanro', 'yksittaisKayttovoima', 'kunta', 'NEDC_Co2', 'NEDC2_Co2', 'WLTP_Co2', 'WLTP2_Co2', 'matkamittarilukema', 'valmistenumero2', 'jarnro']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5122260 entries, 0 to 5122259
Data columns (total 41 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   ajoneuvoluokka             object 
 1   ensirekisterointipvm       object 
 2   ajone

In [8]:
cols_keep = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "variantti",
    "versio",
    "ensirekisterointipvm",
    "kayttoonottopvm",
    "ajoneuvoluokka",
    "ajoneuvoryhma",
    "korityyppi",
    "kayttovoima",
    "yksittaisKayttovoima",
    "sahkohybridi",
    "sahkohybridinluokka",
    "iskutilavuus",
    "suurinNettoteho",
    "sylintereidenLkm",
    "omamassa",
    "vaihteisto",
    "vaihteidenLkm",
    "ahdin",
    "ovienLukumaara",
    "istumapaikkojenLkm",
    "voimanvalJaTehostamistapa",
    "matkamittarilukema"
]

traficom_reduced = df[cols_keep].copy()

print(traficom_reduced.shape)
traficom_reduced.head()

traficom_reduced.to_csv("traficom_reduced_raw.csv", index=False)

(5122260, 25)


In [9]:
missing = (
    traficom_reduced.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
)

print(missing)

sahkohybridinluokka          0.924085
vaihteidenLkm                0.588855
sahkohybridi                 0.570312
ajoneuvoryhma                0.542370
ovienLukumaara               0.520365
vaihteisto                   0.486266
ahdin                        0.474071
matkamittarilukema           0.448032
variantti                    0.374765
korityyppi                   0.359669
versio                       0.339221
sylintereidenLkm             0.331299
suurinNettoteho              0.330044
iskutilavuus                 0.265691
kaupallinenNimi              0.231678
istumapaikkojenLkm           0.231567
yksittaisKayttovoima         0.230334
kayttovoima                  0.230329
voimanvalJaTehostamistapa    0.195374
mallimerkinta                0.178238
ensirekisterointipvm         0.030804
omamassa                     0.002126
merkkiSelvakielinen          0.000360
kayttoonottopvm              0.000000
ajoneuvoluokka               0.000000
Name: missing_share, dtype: float64


In [10]:
cat_cols = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "ajoneuvoluokka",
    "ajoneuvoryhma",
    "korityyppi",
    "kayttovoima",
    "yksittaisKayttovoima",
    "sahkohybridi",
    "sahkohybridinluokka",
    "vaihteisto",
    "ahdin",
]

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(traficom_reduced[col].value_counts(dropna=False).head(15))


--- merkkiSelvakielinen ---
merkkiSelvakielinen
Toyota            442877
Volvo             295934
Ford              264593
Mercedes-Benz     263777
Volkswagen, VW    189455
Muuli             180017
Skoda             174880
Volkswagen        172155
BMW               147263
Nissan            143570
Omavalmiste       129396
Audi              125486
Kia               118342
Opel              110207
Honda              92451
Name: count, dtype: int64

--- mallimerkinta ---
mallimerkinta
NaN                                                   912979
TOYOTA COROLLA Farmari (AC) 4ov 1798cm3                17589
XC60 Farmari (AC) 5ov 1969cm3 A                        16893
1250 XI/250                                            16596
TRANSPORTER Umpikorinen (BB) 6ov 1968cm3               16537
TOYOTA RAV4 Farmari (AC) 4ov 2487cm3                   15669
TOYOTA AURIS Monikäyttöajoneuvo (AF) 4ov 1798cm3       15447
Nissan Qashqai Monikäyttöajoneuvo (AF) 4ov 1197cm3     14527
TOYOTA YARIS Viistoperä (

In [11]:
df = traficom_reduced.copy()

# Traficom uses two different raw date formats here:
# ensirekisterointipvm is typically dd.mm.yyyy, while kayttoonottopvm is yyyymmdd.
# The previous generic to_datetime call parsed yyyymmdd integers as Unix ns timestamps,
# which collapsed use_year to 1970 and inflated vehicle_age.
raw_date_sample = df[["ensirekisterointipvm", "kayttoonottopvm"]].head(10).copy()

df["ensirekisterointipvm"] = pd.to_datetime(
    df["ensirekisterointipvm"].astype("string").str.strip(),
    format="%d.%m.%Y",
    errors="coerce",
)

raw_use_date = df["kayttoonottopvm"].astype("string").str.strip()
raw_use_date = raw_use_date.str.replace(r"\\.0$", "", regex=True)
raw_use_date = raw_use_date.where(raw_use_date.str.len() == 8, raw_use_date.str.zfill(8))
raw_use_date = raw_use_date.where(~raw_use_date.str.match(r"^\\d{4}0000$"), raw_use_date.str[:4] + "0101")
df["kayttoonottopvm"] = pd.to_datetime(raw_use_date, format="%Y%m%d", errors="coerce")

num_cols = [
    "iskutilavuus",
    "suurinNettoteho",
    "sylintereidenLkm",
    "omamassa",
    "vaihteidenLkm",
    "ovienLukumaara",
    "istumapaikkojenLkm",
    "matkamittarilukema",
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [12]:
df[date_cols + num_cols].info()
df[num_cols].describe()

NameError: name 'date_cols' is not defined

In [ ]:
text_cols = [
    "merkkiSelvakielinen",
    "mallimerkinta",
    "kaupallinenNimi",
    "variantti",
    "versio",
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

In [ ]:
df["brand"] = df["merkkiSelvakielinen"]
df["model"] = df["mallimerkinta"]

In [ ]:
print(df["brand"].value_counts(dropna=False).head(20))
print(df["model"].value_counts(dropna=False).head(20))

brand
toyota            442880
volvo             295934
ford              264593
mercedes-benz     263777
volkswagen, vw    189455
muuli             180017
skoda             174880
volkswagen        172158
bmw               147263
nissan            143570
omavalmiste       129513
audi              125486
kia               118344
opel              110207
honda              92453
peugeot            87760
aku                86478
respo              78208
valmet             71355
majava             68350
Name: count, dtype: Int64
model
<NA>                                                  912979
toyota corolla farmari (ac) 4ov 1798cm3                17592
xc60 farmari (ac) 5ov 1969cm3 a                        16895
nissan qashqai monikäyttöajoneuvo (af) 4ov 1197cm3     16781
1250 xi/250                                            16596
transporter umpikorinen (bb) 6ov 1968cm3               16559
toyota rav4 farmari (ac) 4ov 2487cm3                   15680
toyota auris monikäyttöajoneuvo (af

In [ ]:
for col in ["ajoneuvoluokka", "ajoneuvoryhma"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- ajoneuvoluokka ---
ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
M3        8596
N2G       7458
T2        6591
L7e       4493
L6e       3546
N3G       2922
L2e       2282
M2        2263
L2        1320
L4         726
Name: count, dtype: int64

--- ajoneuvoryhma ---
ajoneuvoryhma
NaN            2778158
1, 13           603755
74              183494
75, 509         156317
2, 13           145061
109             138175
75               97355
509              94253
13, 20, 513      78761
6, 13            65029
107              62398
75, 933          49134
9, 13            47708
21               44689
7, 13            41542
73               33980
14               26529
13               24920
13, 20           24540
85         

In [ ]:
car_df = df.copy()

car_df = car_df[
    car_df["ajoneuvoluokka"].notna()
].copy()

print(car_df["ajoneuvoluokka"].value_counts(dropna=False).head(20))

ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
Name: count, dtype: int64


In [ ]:
print(car_df["ajoneuvoluokka"].dropna().astype(str).str.strip().value_counts().head(20))

ajoneuvoluokka
M1     2565062
O1      809597
MUU     422404
N1      272680
O2      196956
M1G     119736
T3       80733
MA       62755
L3       62148
L3e      58310
O4       54939
N1G      54728
L1e      53280
MTK      50535
T        49790
N3       46526
T1       45547
L1       32558
N2       30763
LTR      11468
Name: count, dtype: int64


In [ ]:
car_df["ajoneuvoluokka"] = car_df["ajoneuvoluokka"].astype("string").str.strip().str.lower()
m1_df = car_df[car_df["ajoneuvoluokka"] == "m1"].copy()
print(m1_df.shape)

(2565062, 27)


In [ ]:
for col in ["mallimerkinta", "kaupallinenNimi", "variantti", "versio"]:
    print(f"\n--- {col} ---")
    print(m1_df[col].value_counts(dropna=False).head(20))


--- mallimerkinta ---
mallimerkinta
toyota corolla farmari (ac) 4ov 1798cm3                    17552
nissan qashqai monikäyttöajoneuvo (af) 4ov 1197cm3         16779
toyota rav4 farmari (ac) 4ov 2487cm3                       15637
toyota auris monikäyttöajoneuvo (af) 4ov 1798cm3           15455
toyota yaris viistoperä (ab) 4ov 1490cm3                   14348
model y monikäyttöajoneuvo (af) 5ov                        14207
model 3 sedan (aa) 4ov                                     13802
toyota yaris monikäyttöajoneuvo (af) 4ov 1329cm3           13745
toyota c-hr viistoperä (ab) 4ov 1798cm3                    13324
v60 farmari (ac) 5ov 1969cm3 a                             11517
nissan qashqai monikäyttöajoneuvo (af) 4ov 1598cm3         11409
toyota avensis monikäyttöajoneuvo (af) 4ov 1798cm3         11269
fiesta viistoperä (ab) 4ov 998cm3                          10404
passat farmari (ac) 5ov 1395cm3 a                          10250
toyota yaris hybrid monikäyttöajoneuvo (af) 4ov 1497c

In [ ]:
m1_df.loc[~m1_df["iskutilavuus"].between(500, 10000, inclusive="both"), "iskutilavuus"] = pd.NA
m1_df.loc[~m1_df["suurinNettoteho"].between(20, 1000, inclusive="both"), "suurinNettoteho"] = pd.NA
m1_df.loc[~m1_df["sylintereidenLkm"].between(2, 16, inclusive="both"), "sylintereidenLkm"] = pd.NA
m1_df.loc[~m1_df["omamassa"].between(500, 5000, inclusive="both"), "omamassa"] = pd.NA
m1_df.loc[~m1_df["matkamittarilukema"].between(0, 1000000, inclusive="both"), "matkamittarilukema"] = pd.NA
m1_df.loc[~m1_df["ovienLukumaara"].between(2, 6, inclusive="both"), "ovienLukumaara"] = pd.NA
m1_df.loc[~m1_df["istumapaikkojenLkm"].between(1, 9, inclusive="both"), "istumapaikkojenLkm"] = pd.NA

NameError: name 'pd' is not defined

In [ ]:
m1_df["model_family"] = (
    m1_df["kaupallinenNimi"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(m1_df["model_family"].value_counts(dropna=False).head(30))

NameError: name 'm1_df' is not defined

In [ ]:
total_m1 = len(m1_df)
print(total_m1)

2565062


In [ ]:
print(m1_df.columns.tolist())

['merkkiSelvakielinen', 'mallimerkinta', 'kaupallinenNimi', 'variantti', 'versio', 'ensirekisterointipvm', 'kayttoonottopvm', 'ajoneuvoluokka', 'ajoneuvoryhma', 'korityyppi', 'kayttovoima', 'yksittaisKayttovoima', 'sahkohybridi', 'sahkohybridinluokka', 'iskutilavuus', 'suurinNettoteho', 'sylintereidenLkm', 'omamassa', 'vaihteisto', 'vaihteidenLkm', 'ahdin', 'ovienLukumaara', 'istumapaikkojenLkm', 'voimanvalJaTehostamistapa', 'matkamittarilukema', 'brand', 'model', 'model_family']


In [ ]:
registration_year = m1_df["ensirekisterointipvm"].dt.year
use_year = m1_df["kayttoonottopvm"].dt.year
vehicle_year = use_year.fillna(registration_year)

m1_df["registration_year"] = registration_year
m1_df["use_year"] = use_year
m1_df["vehicle_year"] = vehicle_year

print("Raw date sample:")
print(raw_date_sample)
print("\nParsed date sample:")
print(m1_df[["ensirekisterointipvm", "kayttoonottopvm"]].head(10))
print("\nregistration_year.describe()")
print(m1_df["registration_year"].describe())
print("\nuse_year.describe()")
print(m1_df["use_year"].describe())
print("\nvehicle_year.describe() before age calculation")
print(m1_df["vehicle_year"].describe())
print("\nregistration_year value_counts()")
print(m1_df["registration_year"].value_counts(dropna=False).head(20))
print("\nvehicle_year value_counts()")
print(m1_df["vehicle_year"].value_counts(dropna=False).head(20))

In [ ]:
reference_year = 2025
m1_df.loc[~m1_df["vehicle_year"].between(1950, 2025, inclusive="both"), "vehicle_year"] = pd.NA
m1_df["vehicle_age"] = reference_year - m1_df["vehicle_year"]
m1_df.loc[~m1_df["vehicle_age"].between(0, 80, inclusive="both"), "vehicle_age"] = pd.NA

print("\nvehicle_year.describe() after validity filter")
print(m1_df["vehicle_year"].describe())
print("\nvehicle_age.describe()")
print(m1_df["vehicle_age"].describe())

In [ ]:
def make_year_band(y):
    if pd.isna(y):
        return pd.NA
    y = int(y)
    if y < 2005:
        return "pre_2005"
    elif y <= 2009:
        return "2005_2009"
    elif y <= 2014:
        return "2010_2014"
    elif y <= 2019:
        return "2015_2019"
    else:
        return "2020_2025"

m1_df["year_band"] = m1_df["vehicle_year"].apply(make_year_band)

In [ ]:
total_m1 = len(m1_df)

brand_summary = (
    m1_df.groupby("brand")
    .agg(
        total_registered_vehicles=("brand", "size"),
        median_vehicle_age=("vehicle_age", "median"),
        median_engine_cc=("iskutilavuus", "median"),
        median_power_kw=("suurinNettoteho", "median"),
        median_mileage=("matkamittarilukema", "median"),
    )
    .reset_index()
)

brand_summary["brand_share_of_market"] = (
    brand_summary["total_registered_vehicles"] / total_m1
)

brand_summary = brand_summary.sort_values("total_registered_vehicles", ascending=False)
brand_summary.head(20)

In [ ]:
brand_summary.to_csv("traficom_m1_brand_summary.csv", index=False)